In [62]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [63]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [64]:
len(words)

32033

In [65]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [66]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [69]:
C = torch.randn((27, 2))


In [70]:
C[5]

tensor([-1.5348, -1.3077])

In [74]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([-1.5348, -1.3077])

In [77]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [78]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [97]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [98]:
h

tensor([[-0.9748,  0.5114,  0.7484,  ...,  0.4247, -0.9920, -0.3765],
        [-0.9999,  0.9981, -0.1682,  ...,  0.6167, -0.9843,  0.9095],
        [-0.9966, -0.9785, -0.9953,  ...,  0.3233, -0.9965,  0.9908],
        ...,
        [ 0.9997, -0.7267, -0.9046,  ...,  0.6322,  0.1778,  0.2188],
        [-0.9951,  1.0000,  0.9632,  ...,  0.5994, -0.9918,  0.9313],
        [ 1.0000, -1.0000,  0.9589,  ..., -1.0000, -0.0860, -0.9425]])

In [82]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 6])

In [84]:
torch.cat(torch.unbind(emb, 1), 1)

tensor([[-0.9225,  0.4136, -0.9225,  0.4136, -0.9225,  0.4136],
        [-0.9225,  0.4136, -0.9225,  0.4136, -1.5348, -1.3077],
        [-0.9225,  0.4136, -1.5348, -1.3077, -0.0751, -1.6599],
        [-1.5348, -1.3077, -0.0751, -1.6599, -0.0751, -1.6599],
        [-0.0751, -1.6599, -0.0751, -1.6599,  0.7665,  1.2365],
        [-0.9225,  0.4136, -0.9225,  0.4136, -0.9225,  0.4136],
        [-0.9225,  0.4136, -0.9225,  0.4136,  0.1817, -0.4211],
        [-0.9225,  0.4136,  0.1817, -0.4211,  2.7533, -0.7751],
        [ 0.1817, -0.4211,  2.7533, -0.7751, -0.6202, -1.9050],
        [ 2.7533, -0.7751, -0.6202, -1.9050,  0.2487,  0.0295],
        [-0.6202, -1.9050,  0.2487,  0.0295, -0.6202, -1.9050],
        [ 0.2487,  0.0295, -0.6202, -1.9050,  0.7665,  1.2365],
        [-0.9225,  0.4136, -0.9225,  0.4136, -0.9225,  0.4136],
        [-0.9225,  0.4136, -0.9225,  0.4136,  0.7665,  1.2365],
        [-0.9225,  0.4136,  0.7665,  1.2365,  0.2487,  0.0295],
        [ 0.7665,  1.2365,  0.2487,  0.0

In [87]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [88]:
a.shape

torch.Size([18])

In [89]:
a.view(9, 2)

tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15],
        [16, 17]])

In [90]:
a.storage()

/var/folders/8t/xqp_gwh92m3292wyxxbn787c0000gn/T/ipykernel_41568/214256462.py:1: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  a.storage()


 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [99]:
W2 = torch.randn(100, 27)
b2 = torch.randn(27)

In [100]:
logits = h @ W2 + b2

In [101]:
counts = logits.exp()

In [102]:
prob = counts / counts.sum(1, keepdim=True)

In [104]:
loss = -prob[torch.arange(32), Y].log().mean()

tensor([1.9287e-12, 7.4548e-08, 2.3920e-07, 3.9376e-04, 1.9705e-10, 9.9806e-09,
        5.6874e-01, 7.0029e-11, 1.3359e-10, 1.4947e-09, 2.3584e-04, 1.5694e-07,
        8.4545e-10, 2.3222e-04, 1.5923e-11, 7.4664e-04, 8.4796e-03, 9.4417e-01,
        9.2427e-02, 7.5350e-07, 1.5644e-05, 1.2174e-06, 1.1860e-11, 4.3183e-07,
        1.3477e-03, 6.6299e-03, 7.3717e-10, 1.1859e-04, 1.9558e-09, 8.3315e-05,
        3.1416e-03, 7.1513e-04])

In [105]:
# cleaned up version

In [106]:
X.shape, Y.shape # dataset

(torch.Size([32, 3]), torch.Size([32]))

In [107]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2))
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
W2 = torch.randn(100, 27)
b2 = torch.randn(27)
parameters = [C, W1, b1, W2, b2]

In [108]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [111]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
#counts = logits.exp()
#prob = counts / counts.sum(1, keepdim=True)
#loss = -prob[torch.arange(32), Y].log().mean()
F.cross_entropy(logits, Y)
loss

tensor(16.0724)